## Merge

In [ ]:
#!/usr/bin/env python3
"""
Web Phishing Dataset Merger
Merges OpenPhish and Tranco web-crawled data into a single compressed CSV file.
"""

import pandas as pd
import os
from pathlib import Path
import logging
from typing import List, Dict, Any
import gzip

# Set up logging
logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s')
logger = logging.getLogger(__name__)

def read_file_content(file_path: Path) -> str:
    """
    Read content from a file. Return 'n/a' if file doesn't exist or can't be read.
    
    Args:
        file_path (Path): Path to the file to read
        
    Returns:
        str: File content or 'n/a' if file doesn't exist
    """
    try:
        if file_path.exists():
            with open(file_path, 'r', encoding='utf-8', errors='ignore') as f:
                content = f.read().strip()
                return content if content else 'n/a'
        else:
            return 'n/a'
    except Exception as e:
        logger.warning(f"Error reading file {file_path}: {e}")
        return 'n/a'

def process_web_folder(folder_path: Path, label: int) -> Dict[str, Any]:
    """
    Process a single web-crawled folder and extract data.
    
    Args:
        folder_path (Path): Path to the web-crawled folder
        label (int): Label for the data (0 for benign, 1 for malicious)
        
    Returns:
        Dict[str, Any]: Dictionary containing extracted data
    """
    folder_name = folder_path.name
    
    # Define file paths
    html_file = folder_path / 'html.txt'
    info_file = folder_path / 'info.txt'
    input_url_file = folder_path / 'input_url.txt'
    
    # Read file contents
    html_content = read_file_content(html_file)
    info_content = read_file_content(info_file)
    input_url_content = read_file_content(input_url_file)
    
    return {
        'folder_name': folder_name,
        'html': html_content,
        'info': info_content,
        'input_url': input_url_content,
        'label': label
    }

def process_dataset_folder(dataset_path: Path, label: int) -> List[Dict[str, Any]]:
    """
    Process all web-crawled folders within a dataset folder.
    
    Args:
        dataset_path (Path): Path to the dataset folder (openphish_5000 or tranco_5000)
        label (int): Label for all data in this folder
        
    Returns:
        List[Dict[str, Any]]: List of processed data dictionaries
    """
    data_list = []
    
    if not dataset_path.exists():
        logger.warning(f"Dataset folder does not exist: {dataset_path}")
        return data_list
    
    # Get all subdirectories (web-crawled folders)
    web_folders = [d for d in dataset_path.iterdir() if d.is_dir()]
    
    logger.info(f"Processing {len(web_folders)} folders in {dataset_path.name}")
    
    for i, web_folder in enumerate(web_folders, 1):
        try:
            data = process_web_folder(web_folder, label)
            data_list.append(data)
            
            # Log progress every 500 folders
            if i % 500 == 0 or i == len(web_folders):
                logger.info(f"Processed {i}/{len(web_folders)} folders from {dataset_path.name}")
                
        except Exception as e:
            logger.error(f"Error processing folder {web_folder}: {e}")
            continue
    
    return data_list

def merge_web_phishing_dataset(base_path: str = "../raw/TR-OP/", output_path: str = "../raw/kpd.csv.gz"):
    """
    Main function to merge the web phishing dataset.
    
    Args:
        base_path (str): Base path to the TR-OP folder
        output_path (str): Output path for the merged dataset
    """
    base_dir = Path(base_path)
    output_file = Path(output_path)
    
    # Ensure output directory exists
    output_file.parent.mkdir(parents=True, exist_ok=True)
    
    # Define dataset folders
    openphish_path = base_dir / "openphish_5000"
    tranco_path = base_dir / "tranco_5000"
    
    logger.info("Starting web phishing dataset merger...")
    logger.info(f"Base directory: {base_dir.absolute()}")
    logger.info(f"Output file: {output_file.absolute()}")
    
    # Check if base directory exists
    if not base_dir.exists():
        raise FileNotFoundError(f"Base directory does not exist: {base_dir.absolute()}")
    
    all_data = []
    
    # Process malicious data (OpenPhish)
    logger.info("Processing malicious data (OpenPhish)...")
    malicious_data = process_dataset_folder(openphish_path, label=1)
    all_data.extend(malicious_data)
    logger.info(f"Collected {len(malicious_data)} malicious samples")
    
    # Process benign data (Tranco)
    logger.info("Processing benign data (Tranco)...")
    benign_data = process_dataset_folder(tranco_path, label=0)
    all_data.extend(benign_data)
    logger.info(f"Collected {len(benign_data)} benign samples")
    
    # Create DataFrame
    logger.info("Creating DataFrame...")
    df = pd.DataFrame(all_data)
    
    # Display dataset statistics
    logger.info(f"Dataset created successfully!")
    logger.info(f"Total samples: {len(df)}")
    logger.info(f"Malicious samples: {len(df[df['label'] == 1])}")
    logger.info(f"Benign samples: {len(df[df['label'] == 0])}")
    logger.info(f"Columns: {list(df.columns)}")
    
    # Check for missing data
    missing_counts = df.isin(['n/a']).sum()
    if missing_counts.any():
        logger.info("Missing data summary:")
        for col, count in missing_counts.items():
            if count > 0:
                logger.info(f"  {col}: {count} missing values")
    
    # Display sample of the data
    logger.info("\nSample of the dataset:")
    print(df.head())
    
    # Save to compressed CSV
    logger.info(f"Saving dataset to {output_file}...")
    df.to_csv(output_file, index=False, compression='gzip')
    
    # Verify file size
    file_size_mb = output_file.stat().st_size / (1024 * 1024)
    logger.info(f"Dataset saved successfully! File size: {file_size_mb:.2f} MB")
    
    return df

def validate_dataset_structure(base_path: str = "../raw/TR-OP/"):
    """
    Validate the dataset structure before processing.
    
    Args:
        base_path (str): Base path to the TR-OP folder
    """
    base_dir = Path(base_path)
    
    print("Validating dataset structure...")
    print(f"Base directory: {base_dir.absolute()}")
    print(f"Base directory exists: {base_dir.exists()}")
    
    if base_dir.exists():
        openphish_path = base_dir / "openphish_5000"
        tranco_path = base_dir / "tranco_5000"
        
        print(f"OpenPhish folder exists: {openphish_path.exists()}")
        print(f"Tranco folder exists: {tranco_path.exists()}")
        
        if openphish_path.exists():
            openphish_folders = [d for d in openphish_path.iterdir() if d.is_dir()]
            print(f"OpenPhish subfolders count: {len(openphish_folders)}")
            
            if openphish_folders:
                sample_folder = openphish_folders[0]
                print(f"Sample OpenPhish folder: {sample_folder.name}")
                print(f"  html.txt exists: {(sample_folder / 'html.txt').exists()}")
                print(f"  info.txt exists: {(sample_folder / 'info.txt').exists()}")
                print(f"  input_url.txt exists: {(sample_folder / 'input_url.txt').exists()}")
        
        if tranco_path.exists():
            tranco_folders = [d for d in tranco_path.iterdir() if d.is_dir()]
            print(f"Tranco subfolders count: {len(tranco_folders)}")
            
            if tranco_folders:
                sample_folder = tranco_folders[0]
                print(f"Sample Tranco folder: {sample_folder.name}")
                print(f"  html.txt exists: {(sample_folder / 'html.txt').exists()}")
                print(f"  info.txt exists: {(sample_folder / 'info.txt').exists()}")
                print(f"  input_url.txt exists: {(sample_folder / 'input_url.txt').exists()}")

if __name__ == "__main__":
    # First validate the structure
    print("=" * 60)
    print("WEB PHISHING DATASET MERGER")
    print("=" * 60)
    
    validate_dataset_structure()
    
    print("\n" + "=" * 60)
    print("STARTING DATASET PROCESSING")
    print("=" * 60)
    
    try:
        # Process the dataset
        df = merge_web_phishing_dataset()
        
        print("\n" + "=" * 60)
        print("PROCESSING COMPLETED SUCCESSFULLY")
        print("=" * 60)
        print(f"Final dataset shape: {df.shape}")
        print(f"Output file: ../raw/kpd.csv.gz")
        
    except Exception as e:
        logger.error(f"Error during processing: {e}")
        print(f"\nError: {e}")
        print("Please check the dataset structure and try again.")

## Clean

In [ ]:
#!/usr/bin/env python3
"""
Web Phishing Dataset Cleaner
Cleans the merged web phishing dataset by:
1. Dropping folder_name and input_url columns
2. Extracting domain names from info column URLs
3. Adding new input_url column with extracted domains
4. Saving cleaned dataset to kpd_clean.csv.gz
"""

import pandas as pd
import re
from pathlib import Path
import logging
from urllib.parse import urlparse
from typing import Optional
import numpy as np

# Set up logging
logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s')
logger = logging.getLogger(__name__)

def extract_domain_from_url(url: str) -> str:
    """
    Extract domain name from a URL string.
    
    Args:
        url (str): URL string (can be http/https or other formats)
        
    Returns:
        str: Extracted domain name or 'n/a' if extraction fails
    """
    if pd.isna(url) or url == 'n/a' or not isinstance(url, str):
        return 'n/a'
    
    url = url.strip()
    if not url:
        return 'n/a'
    
    try:
        # Handle cases where URL might not have protocol
        if not url.startswith(('http://', 'https://', 'ftp://', 'ftps://')):
            # Try to detect if it looks like a domain
            if '.' in url and not url.startswith('www.'):
                url = 'http://' + url
            elif url.startswith('www.'):
                url = 'http://' + url
            else:
                # If it doesn't look like a URL, try to extract domain-like patterns
                domain_pattern = r'([a-zA-Z0-9]([a-zA-Z0-9\-]{0,61}[a-zA-Z0-9])?\.)+[a-zA-Z]{2,}'
                match = re.search(domain_pattern, url)
                if match:
                    return match.group(0)
                else:
                    return 'n/a'
        
        # Parse the URL
        parsed = urlparse(url)
        domain = parsed.netloc
        
        # If netloc is empty, try to extract from path
        if not domain and parsed.path:
            # Sometimes the URL might be malformed
            path_parts = parsed.path.split('/')
            if path_parts and '.' in path_parts[0]:
                domain = path_parts[0]
        
        # Clean up the domain
        if domain:
            # Remove port numbers
            domain = domain.split(':')[0]
            # Remove www. prefix if present
            if domain.startswith('www.'):
                domain = domain[4:]
            return domain if domain else 'n/a'
        else:
            return 'n/a'
            
    except Exception as e:
        logger.debug(f"Error parsing URL '{url}': {e}")
        # Fallback: try regex extraction
        try:
            domain_pattern = r'([a-zA-Z0-9]([a-zA-Z0-9\-]{0,61}[a-zA-Z0-9])?\.)+[a-zA-Z]{2,}'
            match = re.search(domain_pattern, url)
            if match:
                domain = match.group(0)
                # Remove www. prefix if present
                if domain.startswith('www.'):
                    domain = domain[4:]
                return domain
            else:
                return 'n/a'
        except:
            return 'n/a'

def validate_domain_extraction(df: pd.DataFrame, sample_size: int = 10):
    """
    Validate domain extraction by showing samples.
    
    Args:
        df (pd.DataFrame): DataFrame with info and input_url columns
        sample_size (int): Number of samples to show for validation
    """
    logger.info("Validating domain extraction with sample data:")
    
    # Get a sample of non-n/a entries
    valid_entries = df[df['info'] != 'n/a'].head(sample_size)
    
    print("\nSample URL → Domain extractions:")
    print("-" * 80)
    for idx, row in valid_entries.iterrows():
        original_url = row['info']
        extracted_domain = row['input_url']
        # Truncate long URLs for display
        display_url = original_url[:60] + "..." if len(original_url) > 60 else original_url
        print(f"URL: {display_url}")
        print(f"Domain: {extracted_domain}")
        print("-" * 40)

def analyze_domain_extraction_stats(df: pd.DataFrame):
    """
    Analyze statistics about domain extraction.
    
    Args:
        df (pd.DataFrame): DataFrame with input_url column
    """
    total_count = len(df)
    valid_domains = len(df[df['input_url'] != 'n/a'])
    invalid_domains = total_count - valid_domains
    
    logger.info("Domain extraction statistics:")
    logger.info(f"Total entries: {total_count}")
    logger.info(f"Valid domains extracted: {valid_domains} ({valid_domains/total_count*100:.1f}%)")
    logger.info(f"Failed extractions (n/a): {invalid_domains} ({invalid_domains/total_count*100:.1f}%)")
    
    if valid_domains > 0:
        # Show top domains
        domain_counts = df[df['input_url'] != 'n/a']['input_url'].value_counts().head(10)
        logger.info("\nTop 10 most frequent domains:")
        for domain, count in domain_counts.items():
            logger.info(f"  {domain}: {count}")

def clean_web_phishing_dataset(input_path: str = "../raw/kpd.csv.gz", 
                              output_path: str = "../raw/kpd_clean.csv.gz"):
    """
    Main function to clean the web phishing dataset.
    
    Args:
        input_path (str): Path to the input CSV file
        output_path (str): Path for the output cleaned CSV file
    """
    input_file = Path(input_path)
    output_file = Path(output_path)
    
    # Ensure output directory exists
    output_file.parent.mkdir(parents=True, exist_ok=True)
    
    logger.info("Starting web phishing dataset cleaning...")
    logger.info(f"Input file: {input_file.absolute()}")
    logger.info(f"Output file: {output_file.absolute()}")
    
    # Check if input file exists
    if not input_file.exists():
        raise FileNotFoundError(f"Input file does not exist: {input_file.absolute()}")
    
    # Read the dataset
    logger.info("Reading input dataset...")
    df = pd.read_csv(input_file, compression='gzip')
    
    logger.info(f"Original dataset shape: {df.shape}")
    logger.info(f"Original columns: {list(df.columns)}")
    
    # Show original data sample
    logger.info("\nOriginal dataset sample:")
    print(df.head())
    
    # Check if required columns exist
    required_columns = ['folder_name', 'input_url', 'info']
    missing_columns = [col for col in required_columns if col not in df.columns]
    if missing_columns:
        raise ValueError(f"Missing required columns: {missing_columns}")
    
    # Step 1: Drop specified columns
    logger.info("Dropping folder_name and input_url columns...")
    columns_to_drop = ['folder_name', 'input_url']
    existing_columns_to_drop = [col for col in columns_to_drop if col in df.columns]
    
    if existing_columns_to_drop:
        df_cleaned = df.drop(columns=existing_columns_to_drop)
        logger.info(f"Dropped columns: {existing_columns_to_drop}")
    else:
        df_cleaned = df.copy()
        logger.warning("No columns to drop (they may already be missing)")
    
    # Step 2: Extract domain names from info column
    logger.info("Extracting domain names from info column...")
    
    # Show some sample URLs before extraction
    sample_urls = df_cleaned[df_cleaned['info'] != 'n/a']['info'].head(5).tolist()
    logger.info("Sample URLs to process:")
    for i, url in enumerate(sample_urls, 1):
        display_url = url[:80] + "..." if len(url) > 80 else url
        logger.info(f"  {i}. {display_url}")
    
    # Extract domains with progress tracking
    logger.info("Processing URLs to extract domains...")
    tqdm_available = True
    try:
        from tqdm import tqdm
        df_cleaned['input_url'] = [extract_domain_from_url(url) for url in tqdm(df_cleaned['info'], desc="Extracting domains")]
    except ImportError:
        # Fallback without progress bar
        tqdm_available = False
        df_cleaned['input_url'] = df_cleaned['info'].apply(extract_domain_from_url)
        logger.info("Domain extraction completed (install tqdm for progress bar)")
    
    # Step 3: Reorder columns for better organization
    logger.info("Reordering columns...")
    # Put input_url right after info, before html and label
    new_column_order = []
    for col in df_cleaned.columns:
        if col == 'info':
            new_column_order.append(col)
            new_column_order.append('input_url')
        elif col != 'input_url':
            new_column_order.append(col)
    
    df_cleaned = df_cleaned[new_column_order]
    
    # Step 4: Analyze and validate results
    logger.info("Analyzing cleaning results...")
    logger.info(f"Cleaned dataset shape: {df_cleaned.shape}")
    logger.info(f"Cleaned columns: {list(df_cleaned.columns)}")
    
    # Validate domain extraction
    validate_domain_extraction(df_cleaned, sample_size=5)
    analyze_domain_extraction_stats(df_cleaned)
    
    # Show label distribution
    if 'label' in df_cleaned.columns:
        label_dist = df_cleaned['label'].value_counts().sort_index()
        logger.info(f"\nLabel distribution:")
        logger.info(f"Benign (0): {label_dist.get(0, 0)}")
        logger.info(f"Malicious (1): {label_dist.get(1, 0)}")
    
    # Show cleaned data sample
    logger.info("\nCleaned dataset sample:")
    print(df_cleaned.head())
    
    # Step 5: Save cleaned dataset
    logger.info(f"Saving cleaned dataset to {output_file}...")
    df_cleaned.to_csv(output_file, index=False, compression='gzip')
    
    # Verify file size
    file_size_mb = output_file.stat().st_size / (1024 * 1024)
    logger.info(f"Cleaned dataset saved successfully! File size: {file_size_mb:.2f} MB")
    
    return df_cleaned

def test_domain_extraction():
    """
    Test domain extraction function with various URL formats.
    """
    test_urls = [
        "https://www.google.com/search?q=test",
        "http://example.com/path/to/page",
        "https://subdomain.example.org:8080/",
        "www.test.com",
        "test.net",
        "ftp://files.example.com/file.txt",
        "https://very-long-subdomain.test-domain.co.uk/very/long/path",
        "invalid-url",
        "",
        "n/a",
        None,
        "192.168.1.1",  # IP address
        "localhost:3000",
        "https://xn--fsq.xn--0zwm56d"  # IDN domain
    ]
    
    print("Testing domain extraction function:")
    print("=" * 80)
    
    for url in test_urls:
        domain = extract_domain_from_url(url)
        print(f"URL: {str(url):<40} → Domain: {domain}")

if __name__ == "__main__":
    print("=" * 60)
    print("WEB PHISHING DATASET CLEANER")
    print("=" * 60)
    
    # Optional: Test domain extraction function
    test_domain_extraction()
    
    print("\n" + "=" * 60)
    print("STARTING DATASET CLEANING")
    print("=" * 60)
    
    try:
        # Clean the dataset
        df_cleaned = clean_web_phishing_dataset()
        
        print("\n" + "=" * 60)
        print("CLEANING COMPLETED SUCCESSFULLY")
        print("=" * 60)
        print(f"Final cleaned dataset shape: {df_cleaned.shape}")
        print(f"Output file: ../raw/kpd_clean.csv.gz")
        
    except Exception as e:
        logger.error(f"Error during cleaning: {e}")
        print(f"\nError: {e}")
        print("Please check the input file and try again.")

## Check Data

In [1]:
import pandas as pd

df = pd.read_csv('../raw/TR-OP/kpd_clean.csv.gz', compression='gzip')

df.shape

(10000, 4)

In [2]:
df.head()

,html,info,input_url,label
0,"<!DOCTYPE html><html lang=""zh_CN""><head>\n ...",http://count.mail.163.com.asianstonetech.com/q...,count.mail.163.com.asianstonetech.com,1
1,"<!DOCTYPE html><html lang=""zh_CN""><head><meta ...",http://roberthood.net/me/young/quak/bizmail.ph...,roberthood.net,1
2,"<!DOCTYPE html><html lang=""zh_CN""><head>\n<met...",https://shgcvdjcvdkgcdghc.freewww.info/cd99f99...,shgcvdjcvdkgcdghc.freewww.info,1
3,"<!DOCTYPE html PUBLIC ""-//W3C//DTD XHTML 1.0 T...",https://abnamro.credit360.com/csr/site/login.a...,abnamro.credit360.com,1
4,"<!DOCTYPE html><html class="""" dir=""LTR"" lang=""...",http://ghdteuegdj.youdontcare.com/abn/,ghdteuegdj.youdontcare.com,1


In [3]:
df.columns

Index(['html', 'info', 'input_url', 'label'], dtype='object')

In [5]:
# drop info columns
df.drop(columns=['info'], inplace=True, errors='ignore')
df.head()

df.rename(columns={'input_url': 'url'}, inplace=True)

df.head()

,html,url,label
0,"<!DOCTYPE html><html lang=""zh_CN""><head>\n ...",count.mail.163.com.asianstonetech.com,1
1,"<!DOCTYPE html><html lang=""zh_CN""><head><meta ...",roberthood.net,1
2,"<!DOCTYPE html><html lang=""zh_CN""><head>\n<met...",shgcvdjcvdkgcdghc.freewww.info,1
3,"<!DOCTYPE html PUBLIC ""-//W3C//DTD XHTML 1.0 T...",abnamro.credit360.com,1
4,"<!DOCTYPE html><html class="""" dir=""LTR"" lang=""...",ghdteuegdj.youdontcare.com,1


In [6]:
df.to_csv('../raw/TR-OP/kpd.csv.gz', index=False, compression='gzip')